# ⭐ RigelSLM — Treino no Google Colab (via Drive)

Este notebook monta seu Google Drive e executa o **treino SFT** do RigelSLM
com GPU (T4 ou superior). Os arquivos (scripts + tokenizer + dados) ficam no
Drive em `rigelllm/`; o notebook trabalha **direto de lá** (sem copiar).

**Versão:** 1.1.0 · **Data:** 04/08/2026
**Pipeline:** `treinar_com_jsonl.py` (SFT — loss só no assistant, métrica corrigida)
**Requer no Drive:** `rigelllm/` com os scripts + `tokenizer/tokenizer.json` + `dados/processed/jsonl/` (dados validados)

## 1. Verificar Ambiente

Verifica se está rodando no Colab e detecta a GPU disponível.

In [3]:
# 1. Verificar ambiente e GPU
import os, sys, subprocess, json, time
from datetime import datetime

# Detecta se está no Colab
try:
    import google.colab
    IS_COLAB = True
    print("✅ Ambiente: Google Colab")
except ImportError:
    IS_COLAB = False
    print("❌ Este notebook foi feito para o Google Colab.")
    sys.exit(1)

# Detecta GPU (só aceita T4 ou superior)
GPU_OK = False
GPU_NOME = ""
try:
    gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
    if gpu_info:
        GPU_NOME = gpu_info[0].strip()
        print(f"🎮 GPU detectada: {GPU_NOME}")
        GPU_OK = any(n in GPU_NOME for n in ["T4", "V100", "A100", "L4", "RTX", "P100"])
        if not GPU_OK:
            print("⚠️  GPU muito fraca. RigelSLM precisa de no mínimo T4.")
            print("   Vá em: Runtime → Alterar tipo de execução → T4 GPU")
    else:
        print("⚠️  Nenhuma GPU detectada.")
        print("   Vá em: Runtime → Alterar tipo de execução → T4 GPU")
except:
    print("⚠️  Não foi possível detectar GPU.")

if not GPU_OK:
    print("\n" + "=" * 60)
    print("   ❌ TREINO CANCELADO: GPU inadequada ou ausente")
    print("   Ative T4 em: Runtime → Alterar tipo de execução")
    print("=" * 60)
    sys.exit(1)

print("✅ GPU OK. Pronto para treinar!")

# Ajusta workers para ambiente Colab
NUM_WORKERS = 2

❌ Este notebook foi feito para o Google Colab.


SystemExit: 1

## 2. Montar Google Drive

Monta o Drive para persistir checkpoints, logs e dados.

In [ ]:
# 2. Montar Google Drive e ir para a pasta do projeto
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/rigelllm"
os.chdir(DRIVE_PATH)

print(f"📁 Diretório de trabalho: {os.getcwd()}")
print(f"📄 Arquivos: {os.listdir()[:15]}...")
print()
print("📂 Pastas encontradas:")
for item in sorted(os.listdir()):
    if os.path.isdir(item):
        qtd = len(os.listdir(item)) if os.path.exists(item) else 0
        print(f"   📁 {item}/ ({qtd} arquivos)")
    else:
        print(f"   📄 {item}")

## 3. Sincronizar arquivos (direto do Drive)

In [ ]:
## 3. Trabalhar direto no Drive

Os arquivos do projeto (scripts, tokenizer, dados) estão no Drive em `rigelllm/`.
O treino será executado **diretamente do Drive** — sem copiar nada para o Colab.

> ⚠️ Os checkpoints/modelos são salvos em `rigelllm/modelo/` no seu Drive.
> Se o Colab desconectar, monte o Drive de novo e retome com `--resume`.

## 4. Instalar Dependências

Instala todas as bibliotecas necessárias para o treino.

In [ ]:
# 4. Instalar dependências (só o necessário para o treino)
print("📦 Instalando dependências...")
!pip install --upgrade pip -q
# Treino SFT precisa de: torch, tokenizers, psutil, tqdm (o treino.py importa esses)
!pip install torch tokenizers psutil tqdm -q
print("✅ Dependências instaladas")

import torch
print(f"🔥 PyTorch {torch.__version__}")
print(f"💠 CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
from tokenizers import Tokenizer
print("✅ Tokenizers OK")

## 5. Configurar Parâmetros do Treino

Ajuste os parâmetros abaixo conforme necessário.

In [ ]:
# 5. Configurar parâmetros do treino SFT
# Altere conforme sua necessidade ↓

# ─── DADOS (treino SÓ com dados validados em processed) ───
PASTA_DADOS = "dados/processed/jsonl"   # base de dados validados
MAX_ARQUIVOS = 5                        # 0 = todos (CUIDADO: Guará tem 941 arquivos)

# ─── TREINO (SFT) ───
EPOCHS = 3                              # 2-3 épocas já dá p/ testar
BATCH_SIZE = 8
SEQ_LEN = 512
LR = 1e-4                               # SFT usa LR menor que o causal
ACCUM = 4                               # acumulação de gradiente
RESUME = True                           # continuar do checkpoint_jsonl.pt?

# ─── OPCIONAL ───
TESTAR_APOS = True                      # testar o modelo com 1 pergunta após treinar
CONVERTER_GGUF = False                  # converter para GGUF ao final?

print("📋 Parâmetros configurados:")
print(f"   📁 Dados: {PASTA_DADOS} | max-arquivos: {MAX_ARQUIVOS}")
print(f"   🔄 Épocas: {EPOCHS} | batch: {BATCH_SIZE} | seq: {SEQ_LEN}")
print(f"   🎯 LR: {LR} | accum: {ACCUM} | resume: {RESUME}")

## 6. Executar Treinamento

Inicia o treino com os parâmetros configurados acima.

In [ ]:
# 6. Executar treino SFT (treinar_com_jsonl.py — o pipeline atual)
print("=" * 60)
print("   🧠 INICIANDO TREINO SFT")
print("=" * 60)

if not os.path.exists("treinar_com_jsonl.py"):
    print("❌ treinar_com_jsonl.py não encontrado no Drive!")
    sys.exit(1)

cmd = [
    "python", "-u", "treinar_com_jsonl.py",
    "--dados", PASTA_DADOS,
    "--no-interactive",
]
if MAX_ARQUIVOS and MAX_ARQUIVOS > 0:
    cmd += ["--max-arquivos", str(MAX_ARQUIVOS)]
cmd += ["--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--seq-len", str(SEQ_LEN),
        "--lr", str(LR),
        "--accum", str(ACCUM)]
if RESUME:
    cmd.append("--resume")

print(f"🚀 Comando: {' '.join(cmd)}")
print()

# Executa com saída em tempo real
processo = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for linha in processo.stdout:
    print(linha, end="")

processo.wait()
print()
if processo.returncode == 0:
    print("✅ Treino concluído com sucesso!")
else:
    print(f"❌ Treino encerrou com código {processo.returncode}")

## 7. (Opcional) Testar o modelo treinado

Gera 1 resposta com `chat.py` (modo one-shot) para conferir se o modelo responde.

In [ ]:
# 7. Testar o modelo treinado (1 pergunta)
if TESTAR_APOS:
    !python chat.py --one-shot "Olá bom dia! Como você está?" --no-stream
    print()
    print("💡 Se a resposta estiver coerente, ótimo! Baixe/use modelo/modelo_melhor.pt.")
else:
    print("⏭️ Teste desabilitado (TESTAR_APOS=False).")

## 8. Resultados já no Drive

In [ ]:
# 8. Resultados já estão no Drive
print("📁 Como estamos trabalhando diretamente no Drive,")
print("   os checkpoints e logs já estão sendo salvos em:")
print(f"   {DRIVE_PATH}/modelo/")
print(f"   {DRIVE_PATH}/logs/")
print()
print("📦 Arquivos gerados (se o treino concluiu):")
if os.path.exists("modelo/modelo_melhor.pt"):
    mb = os.path.getsize("modelo/modelo_melhor.pt") / 1e6
    print(f"   ✅ modelo/modelo_melhor.pt ({mb:.0f} MB)")
if os.path.exists("modelo/checkpoint_jsonl.pt"):
    mb = os.path.getsize("modelo/checkpoint_jsonl.pt") / 1e6
    print(f"   ✅ modelo/checkpoint_jsonl.pt ({mb:.0f} MB)")
if os.path.exists("modelo/modelo.pt"):
    mb = os.path.getsize("modelo/modelo.pt") / 1e6
    print(f"   ✅ modelo/modelo.pt ({mb:.0f} MB)")
print()
print("💡 Depois do treino, baixe modelo/modelo_melhor.pt e teste no seu")
print("   computador (chat.py ou dashboard).")

## 9. (Opcional) Converter para GGUF

Se `CONVERTER_GGUF = True` (na célula 5), converte `modelo/modelo_melhor.pt` para GGUF.


In [ ]:
# 9. (Opcional) Converter para GGUF
if CONVERTER_GGUF:
    print("🔄 Convertendo modelo/modelo_melhor.pt → GGUF (Q4_K_M)...")
    !python converter_para_gguf.py --model modelo/modelo_melhor.pt --quant Q4_K_M --modelfile rigelslm
    print()
    print("📦 GGUF gerado em gguf/")
    if os.path.exists("gguf"):
        for f in sorted(os.listdir("gguf")):
            p = os.path.join("gguf", f)
            if os.path.isfile(p):
                mb = os.path.getsize(p) / 1e6
                print(f"   📄 gguf/{f} ({mb:.0f} MB)")
            else:
                print(f"   📁 gguf/{f}/")
else:
    print("⏭️ Conversão GGUF desabilitada (CONVERTER_GGUF=False).")

---
**Fim do notebook.** Como trabalhamos direto do Drive, `modelo/modelo_melhor.pt` e os logs já ficam salvos no Drive (etapa 7). Depois é só baixar o modelo e testar no seu computador (chat.py / dashboard).
